# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.

# Load environment variable using dotenv
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()


True

In [2]:
import dask.dataframe as dd

/var/folders/67/90hxk3wd4v9d4cvjls01q60h0000gp/T/ipykernel_14775/676544213.py:1: DeprecationWarning: The current Dask DataFrame implementation is deprecated. 
In a future release, Dask DataFrame will use new implementation that
contains several improvements including a logical query planning.
The user-facing DataFrame API will remain unchanged.

The new implementation is already available and can be enabled by
installing the dask-expr library:

    $ pip install dask-expr

and turning the query planning option on:

    >>> import dask
    >>> dask.config.set({'dataframe.query-planning': True})
    >>> import dask.dataframe as dd

API documentation for the new implementation is available at
https://docs.dask.org/en/stable/dask-expr-api.html

Any feedback can be reported on the Dask issue tracker
https://github.com/dask/dask/issues 

  import dask.dataframe as dd


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.

# Access PRICE_DATA environment variable
PRICE_DATA = os.getenv('PRICE_DATA')

#Find all parquet files in the PRICE_DATA directory
parquet_files = glob(os.path.join(PRICE_DATA, '*/*.parquet/part.0.parquet'))

print("PRICE_DATA Path:", PRICE_DATA)
print("Parquet Files Found: " + str(len(parquet_files)))

parquet_files


PRICE_DATA Path: ../../05_src/data/prices/
Parquet Files Found: 11207


['../../05_src/data/prices/CTAS/CTAS_2008.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2018.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2011.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2001.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2000.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2010.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2019.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2009.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2012.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2002.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2024.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2003.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2013.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2016.parquet/part.0.parquet',
 '../../05_src/data/prices/CTAS/CTAS_2006.parquet/part.0.parqu

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Adjusted Close:
    
    - `returns`: (Adj Close / Adj Close_lag) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [4]:
# Write your code below.

ddf = dd.read_parquet(parquet_files)

dd_feat = (ddf.groupby('ticker', group_keys=False)
        .apply(lambda x: x.assign(Close_lag_1=x['Close'].shift(1)))
        .assign(returns=lambda x: x['Close'] / x['Close_lag_1'] - 1)
        .assign(hi_lo_range=lambda x: x['High'] - x['Low'])
    )

/var/folders/67/90hxk3wd4v9d4cvjls01q60h0000gp/T/ipykernel_14775/1745381646.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = (ddf.groupby('ticker', group_keys=False)


In [5]:
dd_feat.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,sector,subsector,year,Close_lag_1,returns,hi_lo_range
ticker,,,,,,,,,,,,,
HUM,2014-01-02,102.760002,103.879997,102.300003,102.839996,95.121140,1393500,Health Care,Managed Health Care,2014,NaN,NaN,1.579994
HUM,2014-01-03,102.860001,103.040001,101.580002,101.760002,94.122208,1234400,Health Care,Managed Health Care,2014,102.839996,-0.010502,1.459999
HUM,2014-01-06,102.300003,102.379997,99.959999,100.760002,93.197266,1850200,Health Care,Managed Health Care,2014,101.760002,-0.009827,2.419998
HUM,2014-01-07,98.720001,101.349998,98.500000,100.550003,93.003029,2561700,Health Care,Managed Health Care,2014,100.760002,-0.002084,2.849998
HUM,2014-01-08,100.660004,100.660004,98.949997,99.400002,91.939346,2332700,Health Care,Managed Health Care,2014,100.550003,-0.011437,1.710007
...,...,...,...,...,...,...,...,...,...,...,...,...,...
JNPR,2012-12-24,19.700001,20.070000,19.420000,20.010000,15.593548,1860200,Information Technology,Communications Equipment,2012,20.170000,-0.007933,0.650000
JNPR,2012-12-26,20.010000,20.400000,19.770000,19.889999,15.500030,2943200,Information Technology,Communications Equipment,2012,20.010000,-0.005997,0.629999
JNPR,2012-12-27,19.930000,19.950001,19.430000,19.790001,15.422105,4461300,Information Technology,Communications Equipment,2012,19.889999,-0.005028,0.520000


+ Convert the Dask data frame to a pandas data frame. 
+ Add a rolling average return calculation with a window of 10 days.
+ *Tip*: Consider using `.rolling(10).mean()`.

(3 pt)

In [20]:
# Write your code below.

dff = dd_feat.compute()

In [23]:
df_avg=(dff.groupby(['ticker']).apply(lambda x: x.assign(ma_return=x['returns'].rolling(window=10).mean())))
df_avg

Date        Open        High         Low       Close  \
ticker ticker                                                              
A      A      2000-01-03   56.330471   56.464592   48.193848   51.502148   
       A      2000-01-04   48.730328   49.266811   46.316166   47.567955   
       A      2000-01-05   47.389126   47.567955   43.141991   44.617310   
       A      2000-01-06   44.080830   44.349072   41.577251   42.918453   
       A      2000-01-07   42.247852   47.165592   42.203148   46.494991   
...                  ...         ...         ...         ...         ...   
ZTS    ZTS    2024-10-22  188.410004  189.820007  187.220001  189.509995   
       ZTS    2024-10-23  189.399994  189.979996  187.559998  188.990005   
       ZTS    2024-10-24  187.559998  188.250000  180.059998  181.500000   
       ZTS    2024-10-25  181.490005  182.029999  179.669998  180.009995   
       ZTS    2024-10-28  181.529999  183.100006  180.699997  182.759995   

                Adj Close   Volume       sector  \
ticker ticker                                     
A      A        43.463032  4674353  Health Care   
       A        40.142937  4765083  Health Care   
       A        37.652866  5758642  Health Care   
       A        36.219185  2534434  Health Care   
       A        39.237453  2819626  Health Care   
...                   ...      ...          ...   
ZTS    ZTS     189.509995  1441900  Health Care   
       ZTS     188.990005  1339500  Health Care   
       ZTS     181.500000  4485900  Health Care   
       ZTS     180.009995  2622900  Health Care   
       ZTS     182.759995  1909700  Health Care   

                                    subsector  year  Close_lag_1   returns  \
ticker ticker                                                                
A      A       Life Sciences Tools & Services  2000          NaN       NaN   
       A       Life Sciences Tools & Services  2000    51.502148 -0.076389   
       A       Life Sciences Tools & Services  2000    47.567955 -0.062030   
       A       Life Sciences Tools & Services  2000    44.617310 -0.038076   
       A       Life Sciences Tools & Services  2000    42.918453  0.083333   
...                                       ...   ...          ...       ...   
ZTS    ZTS                    Pharmaceuticals  2024   189.449997  0.000317   
       ZTS                    Pharmaceuticals  2024   189.509995 -0.002744   
       ZTS                    Pharmaceuticals  2024   188.990005 -0.039632   
       ZTS                    Pharmaceuticals  2024   181.500000 -0.008209   
       ZTS                    Pharmaceuticals  2024   180.009995  0.015277   

               hi_lo_range  ma_return  
ticker ticker                          
A      A          8.270744        NaN  
       A          2.950645        NaN  
       A          4.425964        NaN  
       A          2.771820        NaN  
       A          4.962444        NaN  
...                    ...        ...  
ZTS    ZTS        2.600006   0.001204  
       ZTS        2.419998  -0.000564  
       ZTS        8.190002  -0.004585  
       ZTS        2.360001  -0.005301  
       ZTS        2.400009  -0.005126  

[2779690 rows x 14 columns]

In [24]:
print(df_avg.describe())

                                Date          Open          High  \
count                        2779690  2.779690e+06  2.779690e+06   
mean   2013-02-03 03:10:37.682906880  7.934092e+01  8.025723e+01   
min              2000-01-03 00:00:00  3.020800e-02  3.052100e-02   
25%              2007-02-20 00:00:00  2.352716e+01  2.385028e+01   
50%              2013-06-07 00:00:00  4.411000e+01  4.463000e+01   
75%              2019-03-27 00:00:00  8.268000e+01  8.356000e+01   
max              2024-10-28 00:00:00  9.914170e+03  9.964770e+03   
std                              NaN  1.783904e+02  1.804807e+02   

                Low         Close     Adj Close        Volume          year  \
count  2.779690e+06  2.779690e+06  2.779690e+06  2.779690e+06  2.779690e+06   
mean   7.840407e+01  7.935238e+01  7.005533e+01  8.024826e+06  2.012594e+03   
min    2.697900e-02  3.052100e-02  3.052100e-02  0.000000e+00  2.000000e+03   
25%    2.320000e+01  2.353000e+01  1.665000e+01  9.295770e+05  2.007000

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

### Was it necessary to convert to pandas to calculate the moving average return?

No, it is not strictly necessary to convert to Pandas to calculate the moving average return. Dask has support for rolling operations, so we could have calculated it directly within Dask without converting.

### Would it have been better to do it in Dask? Why?

No, in this particular case it would not, because the data fits in memory and the computation is fast.

In addition, using Dask for rolling window operations is not as convenient as doing it in Pandas. With Dask, you should ensure that the partition sizes you choose are large enough to avoid boundary issues, but keep in mind that larger partitions can begin to slow down your computations. The data should also be index-aligned to ensure that it’s sorted in the correct order. Dask uses the index to determine which rows are adjacent to one another, so ensuring proper sort order is critical for the correct execution of any calculations on the data.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [x] Created a branch with the correct naming convention.
- [x] Ensured that the repository is public.
- [x] Reviewed the PR description guidelines and adhered to them.
- [x] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.